In [ ]:
import pandas as pd
nome_saida = "CIRA-CIC-DoHBrw-2020.csv"
df_final= pd.read_csv(nome_saida)

# ------------------------------------------------------------
# Final message
# ------------------------------------------------------------
print("File generated successfully!")
print(f"Total rows: {df_final.shape[0]}")
print(f"Total columns: {df_final.shape[1]}")




# Remove strong identifiers
# List of columns you want to remove
colunas_para_remover = ['SourceIP', 'DestinationIP', 'SourcePort', 'DestinationPort', 'TimeStamp']



# Selects all columns except the listed ones
df_filtrado = df_final.drop(columns=colunas_para_remover)

# Displays the resulting DataFrame
print(df_filtrado)


# --- MAPPING FOR MALICIOUS DETECTION (CLASS 1) ---
mapping = {
    'DoH': 0,
    'NonDoH': 0,
    'Benign': 0,
    'Malicious': 1
}

# Applying the mapping
df_filtrado.iloc[:, -1] = df_filtrado.iloc[:, -1].map(mapping)

# Removing any values that were not mapped and ensuring int32
df_filtrado.dropna(subset=[df_filtrado.columns[-1]], inplace=True)
#y = df.iloc[:, -1].astype('int32').copy()

print("Class distribution (0 = Normal / 1 = Malicious):")
#print(y.value_counts())

In [ ]:
%%writefile ids_engine_blackbox.py
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import multiprocessing as mp
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE


# ------------------------------------------------------------
# MEMORY PROTECTION
# ------------------------------------------------------------
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


# ------------------------------------------------------------
# GENERAL SETTINGS
# ------------------------------------------------------------
N_RUNS = 30
EPSILONS = [0.001, 0.005, 0.01, 0.02, 0.05]
BATCH_SIZE = 32
ADV_BATCH_SIZE = 64
EPOCHS = 100
THRESHOLD = 0.5



# ------------------------------------------------------------
# CIRA PLAUSIBILITY MASK
# ------------------------------------------------------------
# The CIRA dataframe may contain:
# - identifier / metadata columns: SourceIP, DestinationIP, SourcePort,
#   DestinationPort, TimeStamp;
# - 29 numerical traffic features;
# - one target column, usually DoH or Label.
#
# Identifier columns and target columns are not model features and must never
# be included in the adversarial mask.
#
# Mask convention:
# - 1.0 means the feature can be perturbed.
# - 0.0 means the feature is blocked.
#
# Strategy:
# - deny-by-default: all features are blocked initially;
# - only the features explicitly listed in the selected CIRA mask mode are
#   perturbable.
#
# Available modes:
# - "strict": temporal-only mask; most conservative and scientifically safest;
# - "operational": recommended feature-space mask for experiments;
# - "extended": includes ResponseTimeTime* features; use only when the threat
#   model allows partial influence over response latency.

CIRA_EXPECTED_FEATURES = 29

CIRA_IDENTIFIER_COLUMNS = [
    "SourceIP",
    "DestinationIP",
    "SourcePort",
    "DestinationPort",
    "TimeStamp",
]

CIRA_TARGET_COLUMNS = ["DoH", "Label"]

# Recommended default for the CIRA experiments.
CIRA_MASK_MODE = "operational"

CIRA_ALLOWED_FEATURES_STRICT = [
    "Duration",

    "PacketTimeVariance",
    "PacketTimeStandardDeviation",
    "PacketTimeMean",
    "PacketTimeMedian",
    "PacketTimeMode",
    "PacketTimeSkewFromMedian",
    "PacketTimeSkewFromMode",
    "PacketTimeCoefficientofVariation",
]

CIRA_ALLOWED_FEATURES_OPERATIONAL = [
    "Duration",

    "FlowBytesSent",
    "FlowSentRate",
    "FlowBytesReceived",
    "FlowReceivedRate",

    "PacketLengthVariance",
    "PacketLengthStandardDeviation",
    "PacketLengthMean",
    "PacketLengthMedian",
    "PacketLengthMode",
    "PacketLengthSkewFromMedian",
    "PacketLengthSkewFromMode",
    "PacketLengthCoefficientofVariation",

    "PacketTimeVariance",
    "PacketTimeStandardDeviation",
    "PacketTimeMean",
    "PacketTimeMedian",
    "PacketTimeMode",
    "PacketTimeSkewFromMedian",
    "PacketTimeSkewFromMode",
    "PacketTimeCoefficientofVariation",
]

CIRA_ALLOWED_FEATURES_EXTENDED = [
    *CIRA_ALLOWED_FEATURES_OPERATIONAL,

    "ResponseTimeTimeVariance",
    "ResponseTimeTimeStandardDeviation",
    "ResponseTimeTimeMean",
    "ResponseTimeTimeMedian",
    "ResponseTimeTimeMode",
    "ResponseTimeTimeSkewFromMedian",
    "ResponseTimeTimeSkewFromMode",
    "ResponseTimeTimeCoefficientofVariation",
]

CIRA_MASKS = {
    "strict": CIRA_ALLOWED_FEATURES_STRICT,
    "operational": CIRA_ALLOWED_FEATURES_OPERATIONAL,
    "extended": CIRA_ALLOWED_FEATURES_EXTENDED,
}


def get_cira_allowed_features(mask_mode=CIRA_MASK_MODE):
    """
    Returns the list of perturbable CIRA features for the selected mask mode.

    Parameters
    ----------
    mask_mode : str
        One of: "strict", "operational", or "extended".
    """
    if mask_mode not in CIRA_MASKS:
        raise ValueError(
            f"Unknown CIRA_MASK_MODE={mask_mode!r}. "
            f"Valid modes are: {sorted(CIRA_MASKS)}."
        )

    return list(CIRA_MASKS[mask_mode])


def encode_cira_target(y):
    """
    Encodes the CIRA target column into integer labels.

    Supported values include:
    - boolean True/False;
    - numeric 0/1;
    - string variants such as DoH/NonDoH, Benign/Malicious,
      Attack/Normal and True/False.
    """
    y_series = pd.Series(y).copy()

    if pd.api.types.is_bool_dtype(y_series):
        return y_series.astype("int32")

    if pd.api.types.is_numeric_dtype(y_series):
        return y_series.astype("int32")

    normalized = (
        y_series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace("-", "", regex=False)
        .str.replace("_", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

    mapping = {
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0,
        "doh": 1,
        "nondoh": 0,
        "malicious": 1,
        "benign": 0,
        "attack": 1,
        "normal": 0,
    }

    encoded = normalized.map(mapping)

    if encoded.isna().any():
        unknown_values = sorted(
            y_series[encoded.isna()].astype(str).unique().tolist()
        )
        raise ValueError(
            "Unknown CIRA target values found: "
            f"{unknown_values}. Please map them to 0/1 before calling main()."
        )

    return encoded.astype("int32")


def select_cira_features_and_target(df):
    """
    Selects the 29 numerical CIRA traffic features and the binary target.

    Supported inputs:
    1. Full CIRA dataframe:
       identifier columns + 29 numerical features + DoH/Label.
    2. Filtered CIRA dataframe:
       29 numerical features + DoH/Label.
    3. Generic dataframe:
       29 numerical features + target as the last column.

    DoH/Label are response variables. They are never included in X or in the
    plausibility mask.
    """
    df_local = df.copy()

    target_cols_present = [
        col for col in CIRA_TARGET_COLUMNS
        if col in df_local.columns
    ]

    if target_cols_present:
        target_col = target_cols_present[0]
        y = encode_cira_target(df_local[target_col])
        candidate_df = df_local.drop(columns=target_cols_present)
    else:
        target_col = str(df_local.columns[-1])
        y = encode_cira_target(df_local.iloc[:, -1])
        candidate_df = df_local.iloc[:, :-1].copy()

    present_identifiers = [
        col for col in CIRA_IDENTIFIER_COLUMNS
        if col in candidate_df.columns
    ]

    if present_identifiers:
        candidate_df = candidate_df.drop(columns=present_identifiers)

    leaked_targets = [
        col for col in CIRA_TARGET_COLUMNS
        if col in candidate_df.columns
    ]
    if leaked_targets:
        raise ValueError(
            "Target columns found among input features: "
            f"{leaked_targets}. DoH/Label must be response variables only."
        )

    leaked_identifiers = [
        col for col in CIRA_IDENTIFIER_COLUMNS
        if col in candidate_df.columns
    ]
    if leaked_identifiers:
        raise ValueError(
            "Identifier columns found among model features: "
            f"{leaked_identifiers}."
        )

    if candidate_df.shape[1] != CIRA_EXPECTED_FEATURES:
        raise ValueError(
            f"Expected {CIRA_EXPECTED_FEATURES} CIRA input features after "
            f"removing identifiers and target ({target_col}), but got "
            f"{candidate_df.shape[1]}."
        )

    X = candidate_df.apply(pd.to_numeric, errors="coerce")
    y = pd.Series(y, index=df_local.index)

    valid_rows = X.notna().all(axis=1) & y.notna()
    X = X.loc[valid_rows].copy()
    y = y.loc[valid_rows].astype("int32").copy()

    return X, y, list(X.columns)


def build_cira_plausibility_mask(feature_cols, mask_mode=CIRA_MASK_MODE):
    """
    Builds the CIRA plausibility mask over the 29 numerical input features.

    Mask semantics:
    - 1.0 means the feature may be perturbed.
    - 0.0 means the feature is blocked.

    The mask is deny-by-default. Only the features from the selected mask mode
    are enabled.
    """
    feature_cols = list(feature_cols)

    if len(feature_cols) != CIRA_EXPECTED_FEATURES:
        raise ValueError(
            f"Expected {CIRA_EXPECTED_FEATURES} CIRA input features, "
            f"but received {len(feature_cols)}."
        )

    leaked_targets = [
        col for col in CIRA_TARGET_COLUMNS
        if col in feature_cols
    ]
    if leaked_targets:
        raise ValueError(
            "Target columns found in feature_cols: "
            f"{leaked_targets}. DoH/Label must not be part of the mask."
        )

    leaked_identifiers = [
        col for col in CIRA_IDENTIFIER_COLUMNS
        if col in feature_cols
    ]
    if leaked_identifiers:
        raise ValueError(
            "Identifier columns found in feature_cols: "
            f"{leaked_identifiers}."
        )

    allowed_features = get_cira_allowed_features(mask_mode)

    missing_allowed = [
        feature for feature in allowed_features
        if feature not in feature_cols
    ]
    if missing_allowed:
        raise ValueError(
            "The following expected perturbable CIRA features were not found: "
            f"{missing_allowed}"
        )

    mask_series = pd.Series(0.0, index=feature_cols, dtype="float32")
    mask_series.loc[allowed_features] = 1.0

    return mask_series.values.astype("float32"), mask_series


def print_cira_plausibility_mask_report(mask_series):
    """
    Prints a compact report for the CIRA plausibility mask.
    """
    blocked = mask_series[mask_series == 0.0]
    allowed = mask_series[mask_series == 1.0]

    print("\n" + "-" * 70)
    print("[CIRA Plausibility Mask]")
    print(f"Mask mode: {CIRA_MASK_MODE}")
    print(f"Total input features: {len(mask_series)}")
    print(f"Perturbable features: {len(allowed)}")
    print(f"Blocked features: {len(blocked)}")
    print("\nPerturbable feature names:")
    for idx, name in enumerate(allowed.index):
        print(f"  {idx + 1:02d}. {name}")
    print("\nBlocked feature names:")
    for idx, name in enumerate(blocked.index):
        print(f"  {idx + 1:02d}. {name}")
    print("-" * 70)


def configure_gpu():
    """
    Configures dynamic GPU memory growth.
    """
    gpus = tf.config.list_physical_devices("GPU")

    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass


def set_seed(seed):
    """
    Sets seeds for reproducibility.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)



# TARGET MODEL M1
# ------------------------------------------------------------
def build_mtl_model_M1(input_shape):
    """
    Target model M1: CNN (3 layers) + ECA + Transformer infrastructure.
    """
    inputs = layers.Input(shape=input_shape)

    def eca_block(input_tensor):
        channels = input_tensor.shape[-1]

        squeeze = layers.GlobalAveragePooling2D()(input_tensor)
        squeeze = layers.Reshape((1, 1, channels))(squeeze)

        k_size = max(3, int(abs((np.log2(channels) + 1) / 2 + 0.5)))

        squeeze = layers.Conv2D(
            1,
            kernel_size=(1, k_size),
            padding="same",
            activation="sigmoid",
            use_bias=False,
        )(squeeze)

        return layers.Multiply()([input_tensor, squeeze])

    def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.3):
        x = layers.LayerNormalization(epsilon=1e-6)(inputs)

        x = layers.MultiHeadAttention(
            key_dim=head_size,
            num_heads=num_heads,
            dropout=dropout,
        )(x, x)

        x = layers.Dropout(dropout)(x)
        res = x + inputs

        x = layers.LayerNormalization(epsilon=1e-6)(res)
        x = layers.Dense(ff_dim, activation="relu")(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(inputs.shape[-1])(x)

        return x + res

    # Convolutional Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 3 (Backbone expansion for model M1)
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Transition to the Temporal Stage (Transformer)
    x = layers.Reshape((-1, x.shape[-1]))(x)
    x = transformer_encoder(
        x,
        head_size=128,
        num_heads=4,
        ff_dim=256,
        dropout=0.3,
    )

    # Final classifier
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=l2(1e-2))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(1e-2))(x)

    output = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="binary_output",
    )(x)

    model = Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss=BinaryCrossentropy(),
        metrics=["accuracy", "Precision", "Recall", "AUC"],
    )

    return model

# ------------------------------------------------------------
# SURROGATE MODEL
# ------------------------------------------------------------
def build_surrogate_model(input_shape):
    """
    Surrogate model used in the query-based surrogate black-box scenario.
    """
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    squeeze = layers.GlobalAveragePooling2D()(x)
    excitation = layers.Dense(128 // 4, activation="relu")(squeeze)
    excitation = layers.Dense(128, activation="sigmoid")(excitation)
    excitation = layers.Reshape((1, 1, 128))(excitation)

    x = layers.Multiply()([x, excitation])
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)

    output = layers.Dense(1, activation="sigmoid", dtype="float32")(x)

    model = Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=BinaryCrossentropy(),
        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
def get_metrics(y_true, y_pred_proba, threshold=0.5):
    """
    Computes binary classification metrics.
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred_proba = np.asarray(y_pred_proba).reshape(-1)

    y_pred = (y_pred_proba > threshold).astype("int32")

    return {
        "Acc": accuracy_score(y_true, y_pred),
        "Prec": precision_score(y_true, y_pred, zero_division=0),
        "Rec": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_pred_proba),
    }


def calculate_ASR_I(y_true, y_clean_proba, y_adv_proba, threshold=0.5):
    """
    Computes the Attack Success Rate for IDS (ASR_I).

    ASR_I considers only malicious samples that were correctly
    detected as malicious before the attack.
    """
    y_true = np.asarray(y_true).reshape(-1)

    y_clean_pred = (np.asarray(y_clean_proba).reshape(-1) > threshold).astype("int32")
    y_adv_pred = (np.asarray(y_adv_proba).reshape(-1) > threshold).astype("int32")

    originally_malicious = (y_true == 1) & (y_clean_pred == 1)
    successfully_evaded = originally_malicious & (y_adv_pred == 0)

    denominator = np.sum(originally_malicious)

    return np.sum(successfully_evaded) / denominator if denominator > 0 else 0.0


# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------
def reshape_to_2d(data, size):
    """
    Applies padding up to the nearest perfect square and reshapes for Conv2D input.
    """
    pad_size = size**2 - data.shape[1]

    padded = np.pad(
        data,
        pad_width=((0, 0), (0, pad_size)),
        mode="constant",
    )

    return padded.reshape(-1, size, size, 1).astype("float32")


def prepare_splits(X, y, seed):
    """
    Splits the data into training, validation, and test sets.
    Applies SMOTE only to the training set.
    Fits MinMaxScaler only on the training set.
    """
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=seed,
        stratify=y,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=seed,
        stratify=y_temp,
    )

    smote = SMOTE(random_state=seed)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    scaler = MinMaxScaler()

    X_train_scaled = scaler.fit_transform(X_train_res).astype("float32")
    X_val_scaled = scaler.transform(X_val).astype("float32")
    X_test_scaled = scaler.transform(X_test).astype("float32")

    size = int(np.ceil(np.sqrt(X_train_scaled.shape[1])))

    return (
        reshape_to_2d(X_train_scaled, size),
        reshape_to_2d(X_val_scaled, size),
        reshape_to_2d(X_test_scaled, size),
        np.asarray(y_train_res).astype("float32"),
        np.asarray(y_val).astype("float32"),
        np.asarray(y_test).astype("float32"),
        size,
        scaler,
    )


def make_tf_dataset(X_data, y_data, batch_size, shuffle=False, seed=42):
    """
    Creates a batched tf.data.Dataset.
    """
    X_data = X_data.astype("float32")
    y_data = np.asarray(y_data).astype("float32").reshape(-1, 1)

    with tf.device("/CPU:0"):
        dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(X_data), 10000),
            seed=seed,
            reshuffle_each_iteration=True,
        )

    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


# ------------------------------------------------------------
# ADVERSARIAL ATTACKS ON THE SURROGATE
# ------------------------------------------------------------
def fgsm_attack_masked(model, x, y, epsilon, feature_mask_2d, batch_size=256):
    """
    Generates an FGSM attack on the surrogate with a feature mask.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")

    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)

    @tf.function
    def fgsm_step(x_batch_tensor, y_batch_tensor):
        with tf.GradientTape() as tape:
            tape.watch(x_batch_tensor)

            y_pred = model(x_batch_tensor, training=False)

            y_pred_safe = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            logits = tf.math.log(y_pred_safe / (1.0 - y_pred_safe))

            loss = tf.nn.sigmoid_cross_entropy_with_logits(
                labels=y_batch_tensor,
                logits=logits,
            )

        gradient = tape.gradient(loss, x_batch_tensor)
        masked_gradient = gradient * mask_tensor

        x_adv = x_batch_tensor + epsilon * tf.sign(masked_gradient)

        return tf.clip_by_value(x_adv, 0.0, 1.0)

    for start in range(0, len(x), batch_size):
        end = start + batch_size

        x_tensor = tf.convert_to_tensor(
            x[start:end].astype("float32"),
            dtype=tf.float32,
        )

        y_tensor = tf.convert_to_tensor(
            y_array[start:end].reshape(-1, 1),
            dtype=tf.float32,
        )

        x_adv_list.append(fgsm_step(x_tensor, y_tensor).numpy())
        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")


def pgd_attack_masked(model, x, y, epsilon, alpha, steps, feature_mask_2d, batch_size=256):
    """
    Generates a PGD attack on the surrogate with a feature mask.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")

    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)

    @tf.function
    def pgd_step(x_adv_tensor, x_orig_tensor, y_tensor):
        with tf.GradientTape() as tape:
            tape.watch(x_adv_tensor)

            y_pred = model(x_adv_tensor, training=False)

            y_pred_safe = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            logits = tf.math.log(y_pred_safe / (1.0 - y_pred_safe))

            loss = tf.nn.sigmoid_cross_entropy_with_logits(
                labels=y_tensor,
                logits=logits,
            )

        gradient = tape.gradient(loss, x_adv_tensor)
        masked_gradient = gradient * mask_tensor

        x_adv_tensor = x_adv_tensor + alpha * tf.sign(masked_gradient)

        perturbation = tf.clip_by_value(
            x_adv_tensor - x_orig_tensor,
            -epsilon,
            epsilon,
        )

        # Apply the mask to the final perturbation for numerical safety.
        perturbation = perturbation * mask_tensor

        return tf.clip_by_value(x_orig_tensor + perturbation, 0.0, 1.0)

    for start in range(0, len(x), batch_size):
        end = start + batch_size

        x_original = tf.convert_to_tensor(
            x[start:end].astype("float32"),
            dtype=tf.float32,
        )

        x_adv = tf.identity(x_original)

        y_tensor = tf.convert_to_tensor(
            y_array[start:end].reshape(-1, 1),
            dtype=tf.float32,
        )

        for _ in range(steps):
            x_adv = pgd_step(x_adv, x_original, y_tensor)

        x_adv_list.append(x_adv.numpy())
        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")


# ------------------------------------------------------------
# BLACK-BOX ADVERSARIAL EVALUATION
# ------------------------------------------------------------
def evaluate_adversarial_blackbox(
    target_model,
    surrogate_model,
    X_test_2d,
    y_test,
    epsilons,
    y_clean_proba,
    feature_mask_2d,
):
    """
    Generates adversarial examples on the surrogate and evaluates the transfer
    of these examples to the target model, measuring times.

    In this black-box setting, y_test is used only for metric computation.
    Adversarial generation uses pseudo-labels produced by the surrogate.
    """
    results = []

    # Pseudo-labels for adversarial generation.
    # Ground-truth labels (y_test) must not be used by the attacker.
    y_test_pseudo_proba = surrogate_model.predict(
        X_test_2d.astype("float32"),
        batch_size=ADV_BATCH_SIZE,
        verbose=0,
    )

    y_test_pseudo = (
        y_test_pseudo_proba.reshape(-1) > THRESHOLD
    ).astype("float32")

    for eps in epsilons:
        # Time Measurement: FGSM Generation
        start_t = time.perf_counter()
        X_fgsm = fgsm_attack_masked(
            surrogate_model,
            X_test_2d,
            y_test_pseudo,
            eps,
            feature_mask_2d,
            ADV_BATCH_SIZE,
        )
        t_gen_fgsm = time.perf_counter() - start_t

        # Time Measurement: PGD Generation
        start_t = time.perf_counter()
        X_pgd = pgd_attack_masked(
            surrogate_model,
            X_test_2d,
            y_test_pseudo,
            eps,
            eps / 4,
            10,
            feature_mask_2d,
            ADV_BATCH_SIZE,
        )
        t_gen_pgd = time.perf_counter() - start_t

        # Time Measurement: FGSM Inference
        start_t = time.perf_counter()
        y_fgsm_proba = target_model.predict(
            X_fgsm,
            batch_size=ADV_BATCH_SIZE,
            verbose=1,
        )
        t_inf_fgsm = time.perf_counter() - start_t

        # Time Measurement: PGD Inference
        start_t = time.perf_counter()
        y_pgd_proba = target_model.predict(
            X_pgd,
            batch_size=ADV_BATCH_SIZE,
            verbose=1,
        )
        t_inf_pgd = time.perf_counter() - start_t

        fgsm_metrics = get_metrics(y_test, y_fgsm_proba, THRESHOLD)
        fgsm_metrics["ASR_I"] = calculate_ASR_I(
            y_test,
            y_clean_proba,
            y_fgsm_proba,
            THRESHOLD,
        )
        fgsm_metrics["time_gen"] = t_gen_fgsm
        fgsm_metrics["time_inf"] = t_inf_fgsm

        pgd_metrics = get_metrics(y_test, y_pgd_proba, THRESHOLD)
        pgd_metrics["ASR_I"] = calculate_ASR_I(
            y_test,
            y_clean_proba,
            y_pgd_proba,
            THRESHOLD,
        )
        pgd_metrics["time_gen"] = t_gen_pgd
        pgd_metrics["time_inf"] = t_inf_pgd

        results.append(
            {
                "epsilon": eps,
                "FGSM": fgsm_metrics,
                "PGD": pgd_metrics,
            }
        )

        del X_fgsm, X_pgd, y_fgsm_proba, y_pgd_proba
        gc.collect()

    return results


# ------------------------------------------------------------
# EXECUTION OF ONE RUN
# ------------------------------------------------------------
def run_single_experiment(X, y, feature_mask_1d, run_id, seed):
    """
    Executes one complete run of the black-box evaluation.
    """
    tf.keras.backend.clear_session()
    gc.collect()
    set_seed(seed)

    (
        X_train_2d,
        X_val_2d,
        X_test_2d,
        y_train_res,
        y_val,
        y_test,
        size,
        scaler,
    ) = prepare_splits(X, y, seed)

    feature_mask_2d = reshape_to_2d(np.array([feature_mask_1d]), size)[0:1]

    # Shadow data are sampled only for surrogate construction.
    # This sampling does not remove samples from target-model training.
    X_shadow, _, _, _ = train_test_split(
        X_train_2d,
        y_train_res,
        train_size=0.20,
        random_state=seed,
        stratify=y_train_res,
    )

    noise = np.random.normal(0, 0.05, X_shadow.shape).astype("float32")
    X_shadow_ood = np.clip(X_shadow + noise, 0.0, 1.0)

    # Target model M1 is trained using the full balanced training set.
    train_ds = make_tf_dataset(
        X_train_2d,
        y_train_res,
        BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    val_ds = make_tf_dataset(
        X_val_2d,
        y_val,
        BATCH_SIZE,
        shuffle=False,
        seed=seed,
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
        ),
    ]

    target_model = build_mtl_model_M1(input_shape=(size, size, 1))

    start_train = time.perf_counter()

    history = target_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    train_time = time.perf_counter() - start_train
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    y_shadow_pseudo = target_model.predict(
        X_shadow_ood,
        batch_size=ADV_BATCH_SIZE,
        verbose=1,
    )

    shadow_ds = make_tf_dataset(
        X_shadow_ood,
        y_shadow_pseudo,
        BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    surrogate_model = build_surrogate_model(input_shape=(size, size, 1))

    surrogate_callbacks = [
        EarlyStopping(
            monitor="loss",
            patience=5,
            restore_best_weights=True,
        )
    ]

    surrogate_model.fit(
        shadow_ds,
        epochs=30,
        callbacks=surrogate_callbacks,
        verbose=1,
    )

    y_clean_proba = target_model.predict(
        X_test_2d.astype("float32"),
        batch_size=ADV_BATCH_SIZE,
        verbose=1,
    )

    clean_metrics = get_metrics(y_test, y_clean_proba, THRESHOLD)

    y_clean_pred = (y_clean_proba.reshape(-1) > THRESHOLD).astype("int32")
    y_test_flat = np.asarray(y_test).reshape(-1).astype("int32")
    originally_malicious = (y_test_flat == 1) & (y_clean_pred == 1)

    print("\n" + "-" * 60)
    print(f"[Black-Box Diagnostics - Run {run_id}]")
    print("Clean metrics:", clean_metrics)
    print("y_test distribution:", np.bincount(y_test_flat))
    print("Clean prediction distribution:", np.bincount(y_clean_pred))
    print("Correctly detected malicious samples:", originally_malicious.sum())
    print(
        "Clean probabilities - min/max/mean:",
        float(y_clean_proba.min()),
        float(y_clean_proba.max()),
        float(y_clean_proba.mean()),
    )
    print("-" * 60)

    adversarial_metrics = evaluate_adversarial_blackbox(
        target_model,
        surrogate_model,
        X_test_2d,
        y_test,
        EPSILONS,
        y_clean_proba,
        feature_mask_2d,
    )

    # Calculation of total accumulated times per attack
    total_run_time_fgsm = sum((adv["FGSM"]["time_gen"] + adv["FGSM"]["time_inf"]) for adv in adversarial_metrics)
    total_run_time_pgd = sum((adv["PGD"]["time_gen"] + adv["PGD"]["time_inf"]) for adv in adversarial_metrics)

    print(f"\n[Run {run_id} Completed] Target Time = {train_time:.2f}s")
    print(f"Total Accumulated Time in the Run: FGSM={total_run_time_fgsm:.2f}s | PGD={total_run_time_pgd:.2f}s")

    for adv in adversarial_metrics:
        print(
            f"   -> Eps={adv['epsilon']} | "
            f"FGSM (F1={adv['FGSM']['F1']:.4f}, ASR_I={adv['FGSM']['ASR_I']:.4f}, Gen: {adv['FGSM']['time_gen']:.3f}s, Inf: {adv['FGSM']['time_inf']:.3f}s) | "
            f"PGD (F1={adv['PGD']['F1']:.4f}, ASR_I={adv['PGD']['ASR_I']:.4f}, Gen: {adv['PGD']['time_gen']:.3f}s, Inf: {adv['PGD']['time_inf']:.3f}s)"
        )

    result = {
        "run": run_id,
        "seed": seed,
        "best_epoch": best_epoch,
        "train_time": train_time,
        "clean": {
            **clean_metrics,
            "n_detected_malicious_clean": int(originally_malicious.sum()),
        },
        "adversarial": adversarial_metrics,
        "total_time_fgsm": total_run_time_fgsm,
        "total_time_pgd": total_run_time_pgd
    }

    del target_model, surrogate_model, train_ds, val_ds, shadow_ds
    del X_train_2d, X_val_2d, X_test_2d
    del X_shadow, X_shadow_ood
    del y_train_res, y_val, y_test, y_clean_proba

    gc.collect()
    tf.keras.backend.clear_session()

    return result


def _worker_run(run_id, seed, X, y, feature_mask_1d, return_dict):
    """
    Function called by the subprocess.
    """
    configure_gpu()

    result = run_single_experiment(
        X,
        y,
        feature_mask_1d,
        run_id,
        seed,
    )

    return_dict[run_id] = result


# ------------------------------------------------------------
# SUMMARIZATION AND EXPORT
# ------------------------------------------------------------
def summarize_adversarial_results(all_results):
    """
    Prints the mean and standard deviation of adversarial metrics and times.
    """
    print("\n" + "=" * 70)
    print("ADVERSARIAL RESULTS (QUERY-BASED SURROGATE BLACK-BOX): MEAN ± STANDARD DEVIATION")
    print("=" * 70)

    for attack in ["FGSM", "PGD"]:
        print(f"\nAttack: {attack}")

        for eps in EPSILONS:
            print(f"Epsilon = {eps}")

            for metric in ["Acc", "Prec", "Rec", "F1", "AUC", "ASR_I", "time_gen", "time_inf"]:
                values = [
                    adv_result[attack][metric]
                    for r in all_results
                    for adv_result in r["adversarial"]
                    if adv_result["epsilon"] == eps
                ]

                # Handles different formatting for times
                if metric in ["time_gen", "time_inf"]:
                    print(f"  {metric}: {np.mean(values):.4f}s ± {np.std(values):.4f}s")
                else:
                    print(f"  {metric}: {np.mean(values):.4f} ± {np.std(values):.4f}")

    # Display of Total Accumulated Time (All epsilons, per run)
    print("\n" + "=" * 70)
    print("TOTAL ACCUMULATED TIME (GENERATION + INFERENCE FOR ALL EPSILONS)")
    print("=" * 70)

    total_fgsm_times = [r["total_time_fgsm"] for r in all_results]
    total_pgd_times = [r["total_time_pgd"] for r in all_results]

    print(f"  FGSM Total Time: {np.mean(total_fgsm_times):.4f}s ± {np.std(total_fgsm_times):.4f}s")
    print(f"  PGD Total Time:  {np.mean(total_pgd_times):.4f}s ± {np.std(total_pgd_times):.4f}s")
    print("=" * 70)


def export_results_to_csv(
    all_results,
    output_path="adversarial_metrics_query_surrogate_blackbox_CIRA.csv",
):
    """
    Exports clean and adversarial metrics to CSV, now including times.
    """
    rows = []

    for result in all_results:
        run = result["run"]
        seed = result["seed"]
        clean = result.get("clean", {})

        for adv in result["adversarial"]:
            eps = adv["epsilon"]

            for attack in ["FGSM", "PGD"]:
                row = {
                    "run": run,
                    "seed": seed,
                    "condition": "adversarial",
                    "scenario": "query_based_surrogate_blackbox",
                    "attack": attack,
                    "epsilon": eps,
                    **adv[attack],
                    "clean_Acc": clean.get("Acc", np.nan),
                    "clean_Prec": clean.get("Prec", np.nan),
                    "clean_Rec": clean.get("Rec", np.nan),
                    "clean_F1": clean.get("F1", np.nan),
                    "clean_AUC": clean.get("AUC", np.nan),
                    "n_detected_malicious_clean": clean.get(
                        "n_detected_malicious_clean",
                        np.nan,
                    ),
                    "best_epoch": result["best_epoch"],
                    "train_time": result["train_time"],
                    "total_run_time_attack": result["total_time_fgsm"] if attack == "FGSM" else result["total_time_pgd"]
                }

                rows.append(row)

    df_results = pd.DataFrame(rows)
    df_results.to_csv(output_path, index=False)

    print(f"\nResults saved to: {output_path}")

    return df_results


# ------------------------------------------------------------
# MAIN FUNCTION
# ------------------------------------------------------------
def main(df_filtrado, feature_mask_1d=None):
    """
    Executes N_RUNS runs of query-based surrogate black-box evaluation
    with a CIRA plausibility mask.

    Accepted dataframe formats:
    1. Full CIRA dataframe:
       SourceIP, DestinationIP, SourcePort, DestinationPort, TimeStamp,
       29 numeric traffic features, and DoH.
    2. Filtered CIRA dataframe:
       29 numeric traffic features and DoH.
    3. Generic dataframe:
       29 already selected traffic features and the last column as target.

    The DoH column is a response variable and is never included in X or in
    the plausibility mask.
    """
    try:
        mp.set_start_method("spawn")
    except RuntimeError:
        pass

    df = df_filtrado.copy()

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)

    X, y, feature_cols = select_cira_features_and_target(df)

    if feature_mask_1d is None:
        print("\n[Info] No external mask received.")
        print(f"[Info] Building the CIRA plausibility mask from feature names (mode={CIRA_MASK_MODE}).")

        feature_mask_1d, mask_series = build_cira_plausibility_mask(feature_cols)
    else:
        if isinstance(feature_mask_1d, pd.Series):
            feature_mask_1d = feature_mask_1d.reindex(feature_cols).values

        feature_mask_1d = np.asarray(feature_mask_1d, dtype="float32").reshape(-1)

        if len(feature_mask_1d) != X.shape[1]:
            raise ValueError(
                f"feature_mask_1d has size {len(feature_mask_1d)}, "
                f"but X has {X.shape[1]} features."
            )

        if np.isnan(feature_mask_1d).any():
            raise ValueError(
                "feature_mask_1d contains NaN values. If a pandas Series was "
                "provided, make sure its index matches the CIRA feature names."
            )

        invalid_values = set(np.unique(feature_mask_1d)) - {0.0, 1.0}
        if invalid_values:
            raise ValueError(
                "feature_mask_1d must be binary, with values 0.0 or 1.0. "
                f"Found invalid values: {sorted(invalid_values)}"
            )

        mask_series = pd.Series(feature_mask_1d, index=feature_cols, dtype="float32")

    if len(feature_mask_1d) != X.shape[1]:
        raise ValueError(
            f"feature_mask_1d has size {len(feature_mask_1d)}, "
            f"but X has {X.shape[1]} features."
        )

    print_cira_plausibility_mask_report(mask_series)

    all_results = []

    manager = mp.Manager()
    return_dict = manager.dict()

    print("\n" + "=" * 50)
    print(f"STARTING {N_RUNS} RUNS (QUERY-BASED SURROGATE BLACK-BOX)")
    print("=" * 50)

    for run in range(N_RUNS):
        seed = 42 + run

        print(
            f"\n[{time.strftime('%H:%M:%S')}] "
            f"Isolated Process -> Run {run + 1}/{N_RUNS}"
        )

        p = mp.Process(
            target=_worker_run,
            args=(run + 1, seed, X, y, feature_mask_1d, return_dict),
        )

        p.start()
        p.join()

        if (run + 1) in return_dict:
            all_results.append(return_dict[run + 1])
            del return_dict[run + 1]

    summarize_adversarial_results(all_results)

    df_results = export_results_to_csv(all_results)

    return all_results, df_results


In [ ]:
# Running

import importlib
import ids_engine_blackbox

# Reload the module in case you have just edited the file
importlib.reload(ids_engine_blackbox)

# 2. Run the experiment
all_results, df_results = ids_engine_blackbox.main(df_filtrado)

# 3. View the result
df_results.head()